# Week 4 · Part 2 — Scaling, fair comparison, and the errors that matter

**Question:** on real medical data, which of the three classifiers should we trust, and which mistakes does each one make?

Data: Breast Cancer Wisconsin (Diagnostic), 569 tumour samples, 30 measurements each. Target: **1 = malignant, 0 = benign**.

| Word | Meaning |
|---|---|
| scaling | putting every input on a comparable scale (Week 2: `StandardScaler`) |
| confusion matrix | a table of actual class against predicted class |
| recall (malignant) | of all malignant tumours, the share the model found |
| precision (malignant) | of all tumours the model called malignant, the share that really were |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier                  # baseline: always the most frequent class
from sklearn.tree import DecisionTreeClassifier, plot_tree # yes/no questions about one input at a time
from sklearn.neighbors import KNeighborsClassifier         # vote of the k nearest training rows
from sklearn.svm import SVC                                # widest possible margin between classes
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

## 1. The data
In the original file 0 means malignant. We recode it so that **1 = malignant**: the class we most want to find.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)
print("rows, inputs:", X.shape)
print("malignant (1) / benign (0):", y.value_counts().to_dict())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("training rows:", len(X_train), " locked test rows:", len(X_test))

The test rows stay locked until section 6. `StratifiedKFold` keeps the malignant share the same in every fold, as in Week 2.

## 2. Why scale matters: a picture with two inputs
Two inputs with very different ranges:

In [ ]:
pair = ["mean concave points", "mean texture"]
X_train[pair].describe().loc[["min", "max"]].round(3)

**What we see:** `mean concave points` runs from 0 to 0.2, `mean texture` from about 10 to 39. In a Euclidean distance, a difference of 5 in texture outweighs the **whole** range of concave points. KNN will look almost only at texture.

`StandardScaler` rescales each input to mean 0 and standard deviation 1, using training rows only. In a `Pipeline` this happens inside every fold.

In [ ]:
def evaluate(models):
    """Cross-validated accuracy of every model on the same five folds."""
    rows, fold_acc = [], {}
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
        fold_acc[name] = scores
        rows.append({"model": name, "CV accuracy": scores.mean(), "fold SD": scores.std(ddof=1)})
    table = pd.DataFrame(rows).set_index("model").round(3)
    folds = pd.DataFrame(fold_acc, index=[f"fold {i}" for i in range(1, 6)]).T.round(3)
    return table, folds

In [ ]:
X_pair_train = X_train[pair]
knn_raw = KNeighborsClassifier(n_neighbors=15)
knn_scaled = Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=15))])
for name, model in [("KNN, raw", knn_raw), ("KNN, scaled", knn_scaled)]:
    print(name, round(cross_val_score(model, X_pair_train, y_train, cv=cv).mean(), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, model) in zip(axes, [("KNN, raw", knn_raw), ("KNN, scaled", knn_scaled)]):
    model.fit(X_pair_train, y_train)
    c0, c1 = pair
    g0, g1 = np.meshgrid(np.linspace(X_pair_train[c0].min(), X_pair_train[c0].max(), 300),
                         np.linspace(X_pair_train[c1].min(), X_pair_train[c1].max(), 300))
    grid = pd.DataFrame({c0: g0.ravel(), c1: g1.ravel()})
    ax.contourf(g0, g1, model.predict(grid).reshape(g0.shape), alpha=.25, cmap="coolwarm")
    ax.scatter(X_pair_train[c0], X_pair_train[c1], c=y_train, cmap="coolwarm", edgecolor="k", s=12)
    ax.set(title=name, xlabel=c0, ylabel=c1)
plt.show()

**What we see:** without scaling the boundary is almost horizontal: KNN decides by texture alone and reaches only 0.68, barely above the dummy. After scaling it uses both inputs and reaches 0.93. The data did not change, only the distances.

## 3. A fair comparison on all 30 inputs
Six candidates, same folds. Trees ask about one input at a time, so they should not care about scale. KNN and SVM use distances, so they should.

In [ ]:
scaled = lambda model: Pipeline([("scale", StandardScaler()), ("model", model)])
models = {
    "dummy": DummyClassifier(),
    "tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "KNN, raw": KNeighborsClassifier(n_neighbors=5),
    "KNN, scaled": scaled(KNeighborsClassifier(n_neighbors=5)),
    "SVM, raw": SVC(),
    "SVM, scaled": scaled(SVC()),
}
table, folds = evaluate(models)
table

In [ ]:
folds

**What we see:**
- Scaling lifts SVM from 0.91 to 0.97, in every fold, and KNN from 0.93 to 0.96, in four of five folds.
- The tree (0.93) needs no scaling.
- Scaled KNN and scaled SVM are close (0.958 and 0.967). Accuracy alone hardly separates them.

## 4. Which mistakes? The confusion matrix
Accuracy counts all errors the same. Here the two errors are not equal: calling a malignant tumour benign is much worse than the reverse.

`cross_val_predict` gives every training row a prediction from a fold model that did not see it (Week 3). `confusion_matrix` then counts actual against predicted.

In [ ]:
def matrix(model):
    predicted = cross_val_predict(model, X_train, y_train, cv=cv)
    cm = confusion_matrix(y_train, predicted, labels=[1, 0])
    return pd.DataFrame(cm, index=["actual malignant", "actual benign"],
                        columns=["predicted malignant", "predicted benign"]), predicted

for name in ["tree (depth 3)", "KNN, scaled", "SVM, scaled"]:
    print(name); display(matrix(models[name])[0])

**What we see:** the top-right cell counts **missed malignant tumours**: 17 for the tree, 16 for KNN, 8 for SVM. KNN and SVM have almost the same accuracy, yet KNN misses **twice as many** malignant tumours. KNN, in turn, almost never raises a false alarm (2 benign called malignant).

## 5. Precision and recall for the malignant class
Both read straight from the matrix:

$$\text{recall} = \frac{\text{malignant found}}{\text{all actual malignant}} \qquad \text{precision} = \frac{\text{malignant found}}{\text{all predicted malignant}}$$

Recall answers "how many malignant tumours did we catch?"; precision answers "when we say malignant, how often are we right?".

In [ ]:
cm, predicted = matrix(models["SVM, scaled"])
found = cm.loc["actual malignant", "predicted malignant"]
recall_by_hand = found / cm.loc["actual malignant"].sum()
precision_by_hand = found / cm["predicted malignant"].sum()
print("recall by hand:   ", round(recall_by_hand, 3), "  sklearn:", round(recall_score(y_train, predicted), 3))
print("precision by hand:", round(precision_by_hand, 3), "  sklearn:", round(precision_score(y_train, predicted), 3))

In [ ]:
rows = []
for name in ["tree (depth 3)", "KNN, scaled", "SVM, scaled"]:
    cm, predicted = matrix(models[name])
    rows.append({"model": name, "accuracy": accuracy_score(y_train, predicted),
                 "missed malignant": cm.loc["actual malignant", "predicted benign"],
                 "recall (malignant)": recall_score(y_train, predicted),
                 "precision (malignant)": precision_score(y_train, predicted)})
pd.DataFrame(rows).set_index("model").round(3)

**What we see:** SVM has the highest recall (0.95: it finds 95 of every 100 malignant tumours); KNN has the highest precision (0.99) but the lower recall (0.90). For this task a missed malignant tumour is the worse error, so we prefer **SVM, scaled**. Week 5 continues with F1 and thresholds.

## 6. Decide, then open the test rows once
We choose by the evidence above, from training rows only. Then we fit the chosen model on all training rows and evaluate it **once**.

In [ ]:
final = models["SVM, scaled"].fit(X_train, y_train)
test_predicted = final.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, test_predicted), 3))
print("test recall (malignant):", round(recall_score(y_test, test_predicted), 3))
pd.DataFrame(confusion_matrix(y_test, test_predicted, labels=[1, 0]),
             index=["actual malignant", "actual benign"], columns=["predicted malignant", "predicted benign"])

**What we see:** on 143 untouched samples the SVM reaches accuracy 0.97 and recall 0.93: it misses 4 of 53 malignant tumours and raises no false alarm. The test result checks the choice once; as in Week 3, the model is now fixed.

## 7. Strengths and weaknesses, from our evidence

| | Decision tree | KNN | SVM |
|---|---|---|---|
| Needs scaling | no | yes | yes |
| Can be explained | yes: a few readable questions | by listing neighbours | hardly |
| Here (CV accuracy / missed malignant) | 0.93 / 17 | 0.96 / 16 | 0.97 / 8 |
| Risk | overfits when deep | slow with many rows; sensitive to k and scale | sensitive to C, gamma, and scale |

**Practice 4B** repeats this comparison on a different dataset.

**Sources:** AITU Lecture 4, slides 23, 30–35, and 40; ISLP, confusion matrix (physical PDF pages 161–162); UCI Breast Cancer Wisconsin (Diagnostic), DOI 10.24432/C5DW2B, CC BY 4.0.